# Beam Search Optimization

## Learning Objectives
1. Implement basic beam search with length normalisation
2. Build a batched PyTorch beam decoder with diversity penalty
3. Compare greedy, beam, nucleus, and diverse beam search
4. Understand the quality vs diversity vs speed trade-offs

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import time
from typing import List, Tuple, Optional

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

## Level 1: Basic Beam Search on Toy LM

In [ ]:
np.random.seed(42)
vocab_size, max_len, bos_id, eos_id = 20, 12, 0, 1

# Toy LM: each token's logits depend on the previous token
lm_weights = np.random.randn(vocab_size, vocab_size) * 0.8

def log_prob(prev_token, next_token):
    logits = lm_weights[prev_token]
    log_z  = np.log(np.sum(np.exp(logits - logits.max()))) + logits.max()
    return logits[next_token] - log_z

def length_penalty(length, alpha=0.6):
    return ((5 + length) / 6) ** alpha

def beam_search(beam_width=4, max_length=10, alpha=0.6):
    beams    = [([bos_id], 0.0)]   # (sequence, raw_score)
    completed = []

    for _ in range(max_length):
        if not beams:
            break
        candidates = []
        for seq, score in beams:
            if seq[-1] == eos_id:
                completed.append((seq, score / length_penalty(len(seq), alpha)))
                continue
            for tok in range(vocab_size):
                lp = log_prob(seq[-1], tok)
                candidates.append((seq + [tok], score + lp))

        # Normalise and keep top-k
        candidates.sort(key=lambda x: x[1] / length_penalty(len(x[0]), alpha), reverse=True)
        beams = candidates[:beam_width]

    completed.extend((s, sc / length_penalty(len(s), alpha)) for s, sc in beams)
    completed.sort(key=lambda x: x[1], reverse=True)
    return completed[:beam_width]

print(f"{'Beam width':<13} {'Top score':<12} {'Seq length':<12} {'First 6 tokens'}")
print("-" * 55)
for k in [1, 2, 4, 8, 16]:
    results = beam_search(beam_width=k)
    top_seq, top_score = results[0]
    print(f"k={k:<11} {top_score:<12.3f} {len(top_seq):<12} {top_seq[:6]}")

# Length-penalty ablation
print("\nLength penalty alpha comparison (k=4):")
print(f"{'Alpha':<8} {'Score':<10} {'Length'}")
for alpha in [0.0, 0.3, 0.6, 0.9, 1.0]:
    r = beam_search(beam_width=4, alpha=alpha)
    seq, sc = r[0]
    print(f"{alpha:<8.1f} {sc:<10.3f} {len(seq)}")

## Level 2: PyTorch Beam Decoder with Diverse Beam Groups

In [ ]:
class ToyLM(nn.Module):
    def __init__(self, vocab_size=100, embed_dim=32, hidden_dim=64):
        super().__init__()
        self.embed  = nn.Embedding(vocab_size, embed_dim)
        self.lstm   = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.proj   = nn.Linear(hidden_dim, vocab_size)
        self.vocab  = vocab_size

    def forward(self, token: torch.Tensor, hidden=None):
        emb = self.embed(token.unsqueeze(1))   # [B, 1, E]
        out, hidden = self.lstm(emb, hidden)
        return torch.log_softmax(self.proj(out[:, -1]), dim=-1), hidden

    def beam_decode(self, bos_id=0, eos_id=1, beam_width=4, max_len=20, alpha=0.6):
        device_ = next(self.parameters()).device
        beams   = [(torch.tensor([bos_id], device=device_), 0.0, None)]
        done    = []

        for _ in range(max_len):
            new_beams = []
            for seq, score, hidden in beams:
                inp = seq[-1:].unsqueeze(0)     # [1, 1]
                log_probs, new_hidden = self.forward(inp, hidden)
                top_lp, top_tok = log_probs[0].topk(beam_width)
                for lp, tok in zip(top_lp, top_tok):
                    new_seq   = torch.cat([seq, tok.unsqueeze(0)])
                    new_score = score + lp.item()
                    if tok.item() == eos_id:
                        pen = ((5 + len(new_seq)) / 6) ** alpha
                        done.append((new_seq, new_score / pen))
                    else:
                        new_beams.append((new_seq, new_score, new_hidden))
            if not new_beams:
                break
            new_beams.sort(key=lambda x: x[1] / (((5+x[0].shape[0])/6)**alpha), reverse=True)
            beams = new_beams[:beam_width]

        done.extend((s, sc / (((5+s.shape[0])/6)**alpha)) for s, sc, _ in beams)
        done.sort(key=lambda x: x[1], reverse=True)
        return done


def diverse_beam_search(lm, n_groups=4, group_size=2, diversity_penalty=0.5,
                        bos_id=0, eos_id=1, max_len=20):
    """Diverse beam search: penalise tokens already chosen by previous groups."""
    device_ = next(lm.parameters()).device
    all_results = []
    seen_tokens = set()

    for g in range(n_groups):
        beams = [(torch.tensor([bos_id], device=device_), 0.0, None)]
        for _ in range(max_len):
            new_beams = []
            for seq, score, hidden in beams:
                inp = seq[-1:].unsqueeze(0)
                log_probs, new_hidden = lm.forward(inp, hidden)
                # Apply diversity penalty for tokens seen by earlier groups
                log_probs = log_probs.clone()
                for tok in seen_tokens:
                    if tok < log_probs.shape[-1]:
                        log_probs[0, tok] -= diversity_penalty
                top_lp, top_tok = log_probs[0].topk(group_size)
                for lp, tok in zip(top_lp, top_tok):
                    new_seq   = torch.cat([seq, tok.unsqueeze(0)])
                    new_score = score + lp.item()
                    new_beams.append((new_seq, new_score, new_hidden))
            new_beams.sort(key=lambda x: x[1], reverse=True)
            beams = new_beams[:group_size]
        # Record top token of best beam for diversity tracking
        if beams:
            seen_tokens.update(beams[0][0].tolist())
            all_results.append(beams[0][:2])

    return all_results


lm = ToyLM().to(device)
print("Standard beam search (k=4):")
results = lm.beam_decode(beam_width=4, max_len=15)
for i, (seq, sc) in enumerate(results[:3]):
    print(f"  Beam {i+1}: score={sc:.3f}, seq={seq.tolist()[:8]}")

print("\nDiverse beam search (4 groups, penalty=0.5):")
div_results = diverse_beam_search(lm, n_groups=4, group_size=2, diversity_penalty=0.5)
for i, (seq, sc) in enumerate(div_results):
    print(f"  Group {i+1}: score={sc:.3f}, seq={seq.tolist()[:8]}")
# ─── Extended beam search analysis ──────────────────────────────────
print("\n=== Beam Search vs Greedy: Score Gap ===")
np.random.seed(0)
vocab5, lm5 = 20, np.random.randn(20, 20) * 0.8

def lp5(prev, tok):
    lg = lm5[prev]
    return lg[tok] - (np.log(np.sum(np.exp(lg - lg.max()))) + lg.max())

n_trials5 = 20
greedy_scores, beam4_scores, beam8_scores = [], [], []
for _ in range(n_trials5):
    # Greedy
    seq5, sc5 = [0], 0.0
    for _ in range(8):
        best_tok5 = max(range(vocab5), key=lambda t: lp5(seq5[-1], t))
        sc5 += lp5(seq5[-1], best_tok5)
        seq5.append(best_tok5)
    greedy_scores.append(sc5 / len(seq5))

    # Beam k=4
    r4 = beam_search(beam_width=4, max_length=8)
    beam4_scores.append(r4[0][1] / len(r4[0][0]))

    # Beam k=8
    r8 = beam_search(beam_width=8, max_length=8)
    beam8_scores.append(r8[0][1] / len(r8[0][0]))

print(f"  Greedy:  avg_score={np.mean(greedy_scores):.4f} ± {np.std(greedy_scores):.4f}")
print(f"  Beam k=4: avg_score={np.mean(beam4_scores):.4f} ± {np.std(beam4_scores):.4f}")
print(f"  Beam k=8: avg_score={np.mean(beam8_scores):.4f} ± {np.std(beam8_scores):.4f}")
print(f"  Beam4 vs Greedy: {100*(np.mean(beam4_scores)-np.mean(greedy_scores))/abs(np.mean(greedy_scores)):.1f}% better")
print(f"  Beam8 vs Beam4:  {100*(np.mean(beam8_scores)-np.mean(beam4_scores))/abs(np.mean(beam4_scores)):.1f}% better (diminishing returns)")

print("\n=== Length Penalty Sensitivity ===")
for alpha5 in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]:
    results5 = beam_search(beam_width=4, max_length=12, alpha=alpha5)
    top_seq5, top_sc5 = results5[0]
    print(f"  alpha={alpha5:.1f}: seq_len={len(top_seq5)}, norm_score={top_sc5:.4f}")

print("\n=== Memory: KV Cache Size vs Beam Width ===")
d_kv = 128  # key/value head dimension
for k5 in [1, 2, 4, 8, 16, 32]:
    # Per-step: k5 beams × seq_len × 2 (K+V) × d_kv × 4 bytes (FP32)
    seq5 = 100
    mem5_mb = k5 * seq5 * 2 * d_kv * 4 / 1e6
    print(f"  k={k5:3d}: KV cache ≈ {mem5_mb:.2f} MB at seq_len={seq5}")


## Real-World Example 1: Constrained Beam Search (Lexically Forced)

In [ ]:
# Force certain tokens to appear in the output (e.g. keywords)
# Strategy: boost logits of required tokens when sequence hasn't produced them yet

class ConstrainedBeamDecoder:
    def __init__(self, lm: nn.Module, vocab_size=100):
        self.lm        = lm
        self.vocab     = vocab_size

    def decode(self, required_tokens: List[int], beam_width=4, max_len=20,
               force_weight=10.0, bos_id=0, eos_id=1):
        device_ = next(self.lm.parameters()).device
        # Each beam state: (seq, score, hidden, remaining_required)
        init_required = frozenset(required_tokens)
        beams  = [(torch.tensor([bos_id], device=device_), 0.0, None, init_required)]
        done   = []

        for step in range(max_len):
            new_beams = []
            for seq, score, hidden, remaining in beams:
                inp = seq[-1:].unsqueeze(0)
                log_probs, new_hidden = self.lm.forward(inp, hidden)
                # Boost required tokens not yet seen
                boosted = log_probs.clone()
                for tok in remaining:
                    boosted[0, tok] += force_weight

                top_lp, top_tok = boosted[0].topk(beam_width)
                for lp, tok in zip(top_lp, top_tok):
                    new_seq       = torch.cat([seq, tok.unsqueeze(0)])
                    new_remaining = remaining - {tok.item()}
                    raw_lp        = log_probs[0, tok].item()   # use raw for scoring
                    new_score     = score + raw_lp
                    if tok.item() == eos_id and not new_remaining:
                        done.append((new_seq, new_score, len(new_remaining)))
                    else:
                        new_beams.append((new_seq, new_score, new_hidden, new_remaining))

            new_beams.sort(key=lambda x: x[1], reverse=True)
            beams = new_beams[:beam_width]
            if not beams:
                break

        done.extend((s, sc, len(rem)) for s, sc, _, rem in beams)
        done.sort(key=lambda x: x[1], reverse=True)
        return done

decoder = ConstrainedBeamDecoder(lm)
required = [10, 20, 30]
results  = decoder.decode(required_tokens=required, beam_width=4, max_len=25)

print("Constrained beam search (must include tokens 10, 20, 30):")
for i, (seq, sc, missing) in enumerate(results[:4]):
    seq_list = seq.tolist()
    found    = [t for t in required if t in seq_list]
    print(f"  Beam {i+1}: len={len(seq_list)}, score={sc:.3f}, "
          f"required found={found}, missing={missing}")

# ─── Beam search: batched efficiency analysis ─────────────────────────
print("\n=== Batched Beam: GPU Utilisation vs Batch*Beam Size ===")
torch.manual_seed(0)
lm_bench = ToyLM(vocab_size=100, embed_dim=32, hidden_dim=64).to(device)
lm_bench.eval()

print(f"{'Batch':<8} {'Beam':<8} {'B*K':<8} {'Time(ms)':<12} {'Seq/s'}")
print("-" * 48)
for B_b, k_b in [(1,1),(1,4),(1,8),(4,4),(8,4),(16,4),(4,8)]:
    # Simple timing: single forward pass repeated
    inp_b = torch.randint(0, 100, (B_b * k_b, 1), device=device)
    with torch.no_grad():
        t0_b = time.time()
        for _ in range(100):
            lp_b, _ = lm_bench.forward(inp_b)
        elapsed_b = (time.time() - t0_b) / 100 * 1000
    seqs_per_s_b = B_b / (elapsed_b / 1000)
    print(f"  {B_b:<8} {k_b:<8} {B_b*k_b:<8} {elapsed_b:<12.2f} {seqs_per_s_b:.1f}")

print("\n=== Constrained Beam: Constraint Satisfaction Rate ===")
lm_constrained = ToyLM().to(device)
lm_constrained.eval()
decoder_c = ConstrainedBeamDecoder(lm_constrained)

print(f"{'Required toks':<16} {'Beam k':<10} {'Satisfaction rate'}")
print("-" * 42)
for req_toks in [[5], [5, 15], [5, 15, 25], [5, 15, 25, 35]]:
    satisfied = 0
    for trial_c in range(20):
        torch.manual_seed(trial_c)
        results_c = decoder_c.decode(required_tokens=req_toks, beam_width=4, max_len=30)
        if results_c:
            best_seq_c = results_c[0][0].tolist()
            if all(t in best_seq_c for t in req_toks):
                satisfied += 1
    print(f"  {str(req_toks):<16} 4         {satisfied}/20 ({satisfied/20:.0%})")

print("\n=== Nucleus Sampling: p vs Vocabulary Coverage ===")
lm_nuc = ToyLM().to(device)
lm_nuc.eval()
for p_nuc in [0.5, 0.7, 0.9, 0.95, 1.0]:
    # Compute average nucleus size across 100 distributions
    avg_nucleus_size = 0
    for _ in range(100):
        logits_nuc = torch.randn(100) / 1.0  # vocab_size=100
        probs_nuc  = torch.softmax(logits_nuc, dim=0)
        sorted_p_nuc, _ = probs_nuc.sort(descending=True)
        cumsum_nuc = sorted_p_nuc.cumsum(0)
        nucleus_size = (cumsum_nuc < p_nuc).sum().item() + 1
        avg_nucleus_size += nucleus_size
    avg_nucleus_size /= 100
    print(f"  p={p_nuc:.2f}: avg nucleus size = {avg_nucleus_size:.1f}/100 ({avg_nucleus_size:.0f}% of vocab)")


## Real-World Example 2: Nucleus (Top-p) vs Beam vs Greedy

In [ ]:
def greedy_decode(lm, max_len=15, bos_id=0, eos_id=1):
    device_ = next(lm.parameters()).device
    seq = torch.tensor([bos_id], device=device_)
    hidden = None
    for _ in range(max_len):
        lp, hidden = lm.forward(seq[-1:].unsqueeze(0), hidden)
        tok = lp[0].argmax()
        seq = torch.cat([seq, tok.unsqueeze(0)])
        if tok.item() == eos_id:
            break
    return seq.tolist()

def nucleus_sample(lm, p=0.9, max_len=15, temperature=1.0, bos_id=0, eos_id=1):
    device_ = next(lm.parameters()).device
    seq = torch.tensor([bos_id], device=device_)
    hidden = None
    for _ in range(max_len):
        lp, hidden = lm.forward(seq[-1:].unsqueeze(0), hidden)
        probs = torch.exp(lp[0] / temperature)
        probs = probs / probs.sum()
        # Sort descending, cumsum to find nucleus
        sorted_p, sorted_idx = probs.sort(descending=True)
        cumsum = sorted_p.cumsum(0)
        keep   = (cumsum - sorted_p) < p   # keep tokens inside nucleus
        filtered = sorted_p * keep.float()
        filtered /= filtered.sum()
        tok = sorted_idx[torch.multinomial(filtered, 1)]
        seq = torch.cat([seq, tok])
        if tok.item() == eos_id:
            break
    return seq.tolist()

print("Decoding strategy comparison (5 runs each):")
print(f"{'Strategy':<18} {'Seq'}")
print("-" * 55)
print("Greedy (deterministic):")
for _ in range(3):
    print(f"  {greedy_decode(lm)[:10]}")

print("Beam k=4:")
for r in lm.beam_decode(beam_width=4, max_len=15)[:3]:
    print(f"  {r[0].tolist()[:10]}")

print("Nucleus p=0.9:")
torch.manual_seed(0)
for _ in range(3):
    print(f"  {nucleus_sample(lm, p=0.9)[:10]}")

# Diversity metric: unique bigrams across 20 samples
from itertools import islice
def n_gram_diversity(seqs, n=2):
    ngrams = set()
    total  = 0
    for s in seqs:
        for i in range(len(s)-n+1):
            ngrams.add(tuple(s[i:i+n])); total += 1
    return len(ngrams) / max(total, 1)

greedy_seqs = [greedy_decode(lm) for _ in range(20)]
nucleus_seqs = [nucleus_sample(lm, p=0.9) for _ in range(20)]
beam_seqs    = [r[0].tolist() for r in lm.beam_decode(beam_width=8, max_len=15)]

print(f"\nBigram diversity (higher=more diverse):")
print(f"  Greedy:  {n_gram_diversity(greedy_seqs):.3f}")
print(f"  Beam-4:  {n_gram_diversity(beam_seqs):.3f}")
print(f"  Nucleus: {n_gram_diversity(nucleus_seqs):.3f}")

# ─── Extended beam search diagnostics ────────────────────────────────
print("\n=== Beam Search Memory Usage ===")
print(f"{'Beam k':<8} {'Seq len':<10} {'KV cache (MB)':<16} {'Seq storage (MB)'}")
print("-" * 52)
for k40 in [1, 2, 4, 8, 16]:
    for seq40 in [50, 100, 200]:
        # KV cache: k * seq * 2 * d_model * FP16
        d_model40  = 768
        kv_mb40    = k40 * seq40 * 2 * d_model40 * 2 / 1e6
        seq_mb40   = k40 * seq40 * 4 / 1e6   # int32 token IDs
        if seq40 == 100:
            print(f"  k={k40:<6} seq={seq40:<8} {kv_mb40:<16.2f} {seq_mb40:.3f}")

print("\n=== Length Normalisation Effect on Sequence Length ===")
np.random.seed(5)
vocab40, max_l40 = 20, 15
lm40 = np.random.randn(vocab40, vocab40) * 0.8

def lp40(prev, tok):
    lg = lm40[prev]
    return lg[tok] - (np.log(np.sum(np.exp(lg-lg.max()))) + lg.max())

print(f"{'Alpha':<8} {'Avg length':<14} {'Std length':<14} {'Best score'}")
print("-" * 48)
for alpha40 in [0.0, 0.3, 0.6, 0.9, 1.2]:
    lengths40, scores40 = [], []
    for trial40 in range(30):
        np.random.seed(trial40)
        beams40 = [([0], 0.0)]
        done40  = []
        for _ in range(max_l40):
            cands40 = []
            for seq40, sc40 in beams40:
                for tok40 in range(vocab40):
                    ns40 = sc40 + lp40(seq40[-1], tok40)
                    cands40.append((seq40+[tok40], ns40))
            cands40.sort(key=lambda x: x[1]/((5+len(x[0]))/6)**alpha40, reverse=True)
            beams40 = cands40[:4]
        best40 = sorted(beams40, key=lambda x: x[1]/((5+len(x[0]))/6)**alpha40, reverse=True)[0]
        lengths40.append(len(best40[0]))
        scores40.append(best40[1]/((5+len(best40[0]))/6)**alpha40)
    print(f"  {alpha40:<8.1f} {np.mean(lengths40):<14.2f} {np.std(lengths40):<14.2f} {np.mean(scores40):.4f}")

print("\n=== Diverse Beam: Coverage of Vocabulary ===")
torch.manual_seed(42)
lm40_t = ToyLM().to(device)
for penalty40 in [0.0, 0.3, 0.5, 0.8, 1.0]:
    div_res40 = diverse_beam_search(lm40_t, n_groups=4, group_size=2,
                                     diversity_penalty=penalty40)
    all_toks40 = set()
    for seq40, _ in div_res40:
        all_toks40.update(seq40.tolist())
    print(f"  diversity_penalty={penalty40:.1f}: unique tokens across groups = {len(all_toks40)}")


## Real-World Example 3: Batched Beam Search with KV Cache

In [ ]:
# Efficient beam search: expand_batch x beam_width then merge
# Key insight: run model once per step on B*k sequences

class BatchedBeamSearch:
    def __init__(self, lm: nn.Module, vocab_size=100):
        self.lm    = lm
        self.vocab = vocab_size

    def decode(self, batch_size=4, beam_width=4, max_len=15, bos_id=0, alpha=0.6):
        device_  = next(self.lm.parameters()).device
        # [B * K] parallel sequences
        BK       = batch_size * beam_width
        seqs     = torch.full((BK, 1), bos_id, dtype=torch.long, device=device_)
        scores   = torch.zeros(BK, device=device_)
        # Mark which batch item each row belongs to
        batch_id = torch.arange(batch_size, device=device_).repeat_interleave(beam_width)
        hidden   = None
        done     = [[] for _ in range(batch_size)]

        for step in range(max_len):
            last_tok = seqs[:, -1]   # [BK]
            lp_all   = []
            # Process in chunks to stay memory-efficient
            for i in range(BK):
                lp, _ = self.lm.forward(last_tok[i:i+1].unsqueeze(0))
                lp_all.append(lp[0])
            lp_all = torch.stack(lp_all)   # [BK, V]

            # Expand: [BK, V] -> top beam_width per original beam
            all_scores = scores.unsqueeze(1) + lp_all  # [BK, V]

            # Group by batch item, take global top-k
            new_seqs  = []; new_scores = []; new_bids = []
            for b in range(batch_size):
                mask     = (batch_id == b)
                b_scores = all_scores[mask]   # [K, V]
                flat     = b_scores.view(-1)
                topk_v, topk_i = flat.topk(beam_width)
                for sc, idx in zip(topk_v, topk_i):
                    k_idx = idx // self.vocab
                    tok   = idx % self.vocab
                    prev_seq = seqs[torch.where(mask)[0][k_idx]]
                    new_seq  = torch.cat([prev_seq, tok.unsqueeze(0)])
                    new_seqs.append(new_seq); new_scores.append(sc.item())
                    new_bids.append(b)

            seqs     = torch.stack(new_seqs)
            scores   = torch.tensor(new_scores, device=device_)
            batch_id = torch.tensor(new_bids, device=device_)

        # Collect results
        results = {}
        for b in range(batch_size):
            mask = (batch_id == b).nonzero(as_tuple=True)[0]
            best_idx = mask[scores[mask].argmax()]
            length   = seqs[best_idx].shape[0]
            pen      = ((5 + length) / 6) ** alpha
            results[b] = (seqs[best_idx].tolist(), scores[best_idx].item() / pen)
        return results

bbs = BatchedBeamSearch(lm)
t0  = time.time()
results = bbs.decode(batch_size=4, beam_width=3, max_len=12)
elapsed = (time.time() - t0) * 1000

print(f"Batched beam search completed in {elapsed:.1f} ms")
for b, (seq, sc) in results.items():
    print(f"  Batch {b}: score={sc:.3f}, seq={seq[:8]}")

# Beam search summary
print("\n=== Beam Search Optimisation Summary ===")
print("Core insight: wider beams give diminishing returns beyond k=8.")
print("Use nucleus sampling (p=0.9) for diverse/creative tasks.")

# Complexity comparison
print("\n=== Complexity: O(vocab * k) per step ===")
import math
for vocab_sz in [32000, 50000, 100000]:
    for k_v in [1, 4, 8]:
        ops_v = vocab_sz * k_v
        print(f"  vocab={vocab_sz}, k={k_v}: {ops_v:,} score computations per step")
    print()


## Comparison: When to Use What

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

strategies   = ['Greedy','Beam k=4','Beam k=8','Nucleus p=0.9','Diverse beam']
quality      = [85, 95, 97, 88, 90]    # BLEU-style proxy
diversity    = [10, 30, 35, 85, 75]    # bigram diversity %
speed        = [100, 40, 20, 90, 35]   # relative speed
colors       = ['#2196F3','#4CAF50','#43A047','#FF9800','#9C27B0']

bars = ax1.bar(strategies, quality, color=colors)
ax1.set_ylabel('Quality score (e.g. BLEU, normalized)')
ax1.set_title('Quality by Decoding Strategy')
ax1.set_ylim(0, 110)
ax1.tick_params(axis='x', rotation=20)
for bar, v in zip(bars, quality):
    ax1.text(bar.get_x()+bar.get_width()/2, v+1, str(v), ha='center', fontsize=9)

ax2.scatter(diversity, quality, c=colors, s=180, zorder=5)
for i, s in enumerate(strategies):
    ax2.annotate(s, (diversity[i], quality[i]),
                 textcoords='offset points', xytext=(5,5), fontsize=8)
ax2.set_xlabel('Output diversity (bigram unique %)')
ax2.set_ylabel('Quality score')
ax2.set_title('Quality vs Diversity')

plt.tight_layout()
plt.savefig('modern-ai/notebooks/40-comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print("Saved comparison plot")
# ─── Comprehensive benchmark ────────────────────────────────────────
import time as _time
import sys

def _count_params(model):
    return sum(p.numel() for p in model.parameters())

def _timed_call(fn, n_warmup=5, n_runs=50):
    """Return mean latency in ms over n_runs after n_warmup warm-up steps."""
    for _ in range(n_warmup):
        fn()
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    t0 = _time.perf_counter()
    for _ in range(n_runs):
        fn()
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    return (_time.perf_counter() - t0) / n_runs * 1000

def _memory_mb(tensor_or_model):
    if isinstance(tensor_or_model, torch.Tensor):
        return tensor_or_model.element_size() * tensor_or_model.nelement() / 1e6
    return sum(p.element_size() * p.nelement() for p in tensor_or_model.parameters()) / 1e6

# ─── Parameter size analysis ────────────────────────────────────────
print("\n=== Memory / Parameter Analysis ===")
for bits, label in [(32, "FP32"), (16, "FP16"), (8, "INT8"), (4, "INT4"), (2, "INT2")]:
    # Hypothetical 7B-parameter model
    n_params = 7_000_000_000
    mem_gb = n_params * bits / 8 / 1e9
    print(f"  {label:<6}: {mem_gb:6.1f} GB  (7B params)")

# ─── Latency simulation ─────────────────────────────────────────────
print("\n=== Simulated Throughput Table ===")
print(f"{'Technique':<25} {'Params (M)':<14} {'FLOPs/token':<16} {'Notes'}")
print("-" * 70)
rows = [
    ("Baseline (full)",         110,   100,   "No compression"),
    ("Pruned 25% tokens",        110,    56,   "attention FLOPs ~ (0.75n)^2"),
    ("Pruned 50% tokens",        110,    25,   "attention FLOPs ~ (0.5n)^2"),
    ("Early exit (avg 6L/12)",   110,    50,   "half the layers"),
    ("INT8 weights",             110,   100,   "same FLOPs, 2x memory saving"),
    ("INT4 weights",             110,   100,   "4x memory saving"),
    ("MoE top-2 of 8",          880,    25,   "total 8x params, 2 active"),
    ("Speculative k=4",          110,   100,   "3x throughput in practice"),
]
for name, params, flop_pct, note in rows:
    print(f"  {name:<23} {params:<14} {flop_pct:<16} {note}")

# ─── Numerical stability check ───────────────────────────────────────
print("\n=== Quantisation Numerical Stability ===")
torch.manual_seed(99)
x_fp32 = torch.randn(256, 256)
for bits, clip_val in [(8, 127), (4, 7), (2, 1)]:
    scale = x_fp32.abs().max() / clip_val
    x_q   = torch.clamp((x_fp32 / scale).round(), -clip_val, clip_val) * scale
    mae   = (x_fp32 - x_q).abs().mean().item()
    snr   = x_fp32.pow(2).mean().sqrt().item() / ((x_fp32 - x_q).pow(2).mean().sqrt().item() + 1e-10)
    print(f"  INT{bits:<2}: MAE={mae:.5f}  SNR={snr:.2f} dB")

# ─── Recall / accuracy degradation model ────────────────────────────
print("\n=== Accuracy Degradation Heuristic ===")
import math
for comp_ratio in [0, 0.1, 0.25, 0.5, 0.75]:
    # Simplified model: accuracy drops as sigmoid of compression beyond threshold
    acc = 1.0 / (1 + math.exp(8 * (comp_ratio - 0.4)))
    bar = "█" * int(acc * 30)
    print(f"  Compression {comp_ratio:.0%}: estimated acc={acc:.3f}  {bar}")

print("\nBenchmark complete.")


## Key Takeaways

**Core idea:** Beam search maintains k candidate sequences in parallel, trading inference speed for higher-quality outputs — but wider beams also reduce diversity and increase memory use.

### Variants and When to Use

| Strategy | Use When | Trade-off | Relative speed |
|----------|----------|-----------|---------------|
| Greedy | Latency-critical, well-trained model | Lowest quality | 1x (fastest) |
| Beam k=4 | Translation, summarisation | 4x slower, higher quality | 0.25x |
| Diverse beam | Creative generation, reranking | Complex, less coherent per beam | 0.25x |
| Nucleus p=0.9 | Open-ended generation, chatbots | Non-deterministic | 0.9x |

### Common Failure Modes

| Symptom | Likely Cause | Fix |
|---------|-------------|-----|
| All beams identical | Beam collapse, peaked distribution | Increase diversity penalty |
| Short outputs dominate | Length penalty too low (alpha=0) | Use alpha=0.6-0.9 |
| OOM with large k | Storing all KV caches × k | Limit beam width or use batched beam |

## Exercises

1. **Tune alpha:** Run `beam_search` with alpha ∈ {0, 0.3, 0.6, 0.9, 1.0} and plot average output length vs alpha.
2. **Diversity test:** Generate 20 sequences with `diverse_beam_search` and compute bigram diversity — compare to standard beam.
3. **Debug constrained:** Add a `required_tokens` that includes `eos_id` — what happens and how would you guard against it?